In [ ]:
import pandas as pd
import os
from pathlib import Path

# Procurar TechChallenge_Fase2 em múltiplos lugares
possible_roots = [
    Path.cwd(),  # CWD atual
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.home() / 'Documents' / 'POS TECH' / 'Challenges' / 'Repositorio do grupo' / 'FIAP_POSTECH_AI_SCIENTIST' / 'TechChallenge_Fase2',
]

root_dir = None
for candidate in possible_roots:
    if (candidate / 'data' / 'samples').exists():
        root_dir = candidate
        break

if root_dir is None:
    raise FileNotFoundError(f'Não encontrei TechChallenge_Fase2! Testei: {possible_roots}')

os.chdir(root_dir)

print(f'✅ Encontrado em: {root_dir}')
print(f'✅ Mudei para: {os.getcwd()}')
print()

samples_dir = 'data/samples'

bronze_indicador = pd.read_parquet(f'{samples_dir}/indicador_municipio.parquet')
bronze_meta = pd.read_parquet(f'{samples_dir}/meta_municipio.parquet')

bronze_indicador['taxa_alfabetizacao'] = pd.to_numeric(bronze_indicador['taxa_alfabetizacao'], errors='coerce')

print('✅ Dados carregados!')
print(f'indicador_municipio: {bronze_indicador.shape}')
print(f'meta_municipio: {bronze_meta.shape}')

In [ ]:
print('\n' + '='*80)
print('🔵 BRONZE')
print('='*80)
print(bronze_indicador[['ano', 'id_municipio', 'taxa_alfabetizacao']].head())

In [ ]:
silver = bronze_indicador.copy()
silver['meta_uf_2030'] = 80
silver['gap_meta_uf_2030'] = silver['taxa_alfabetizacao'] - silver['meta_uf_2030']
silver['atingiu_meta_uf'] = silver['gap_meta_uf_2030'] >= 0

print('\n' + '='*80)
print('🟢 SILVER')
print('='*80)
print(silver[['ano', 'id_municipio', 'taxa_alfabetizacao', 'gap_meta_uf_2030']].head())

In [ ]:
gold = silver[['id_municipio', 'ano', 'taxa_alfabetizacao', 'meta_uf_2030', 'gap_meta_uf_2030', 'atingiu_meta_uf']].copy()
gold['categoria_risco'] = gold['gap_meta_uf_2030'].apply(lambda x: 'critico' if x < -10 else 'alto' if x < -5 else 'moderado' if x < 0 else 'meta_atingida')

print('\n' + '='*80)
print('🟡 GOLD')
print('='*80)
print(gold[['id_municipio', 'taxa_alfabetizacao', 'gap_meta_uf_2030', 'categoria_risco']].head())

In [ ]:
print('\n' + '='*80)
print('📊 QUERY 1: Top 10 - MAIOR taxa')
print('='*80)
q1 = gold[gold['ano'] == gold['ano'].max()][['id_municipio', 'taxa_alfabetizacao', 'gap_meta_uf_2030']].sort_values('taxa_alfabetizacao', ascending=False).head(10).reset_index(drop=True)
print(q1)

In [ ]:
print('\n' + '='*80)
print('📊 QUERY 3: Top 10 - ABAIXO da meta')
print('='*80)
q3 = gold[(gold['atingiu_meta_uf'] == False) & (gold['ano'] == gold['ano'].max())][['id_municipio', 'taxa_alfabetizacao', 'gap_meta_uf_2030']].sort_values('gap_meta_uf_2030').head(10).reset_index(drop=True)
print(q3)

In [ ]:
print('\n✅ TUDO FUNCIONANDO!')
print(f'Total de registros: {len(gold):,}')